In [ ]:
# Resolve which run to post-process. Every cell below reads its diagnostics
# (diags/..., warpx_used_inputs) relative to the current directory, so cd into
# the run dir first. Runs live under the configured output base (LOCAL_OUTPUT_DIR
# in .env), grouped per deck: OUTPUT_DIR/coil_2d/run_<timestamp>/.
from warpx_polywell.post.reader import chdir_to_run

# Latest coil_2d run. For a specific run instead, pass its DB id:
#   run = chdir_to_run("coil_2d", run_id=<id>)
run = chdir_to_run("coil_2d")
print("post-processing:", run)

In [ ]:
from openpmd_api import Series, Access
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
series = Series("diags/diag/openpmd_%T.h5", Access.read_only)
its = sorted(series.iterations)
it = series.iterations[its[-1]]      # iteration 10

B = it.meshes["B"]
Bx = B["x"].load_chunk()
By = B["y"].load_chunk()
Bz = B["z"].load_chunk()
series.flush()
#B2 = Bx**2 + By**2 + Bz**2

shape  = B["x"].shape            # cells per axis, e.g. [Nz, Nr] in RZ
dx     = B.grid_spacing          # cell size along each axis
x0     = B.grid_global_offset    # lower corner of the grid
axes   = B.axis_labels           # e.g. ('z', 'r') or ('x','y','z')
unit   = B.grid_unit_SI          # multiply spacing/offset by this for meters
geom   = B.geometry              # cartesian / thetaMode / etc.

# physical extents (in SI meters)
lo = [o * unit for o in x0]
hi = [(o + n * d) * unit for o, n, d in zip(x0, shape, dx)]

coords = [x0[i]*unit + (np.arange(shape[i]) + B["x"].position[i]) * dx[i] * unit for i in range(len(shape))][0]

In [ ]:
print(coords.shape)
print(Bx.shape)

In [ ]:
mag = np.hypot(Bx, Bz)

fig, ax = plt.subplots(figsize=(6, 5))
strm = ax.streamplot(
    coords, coords, Bx, Bz,
    color=mag, cmap="viridis", density=1.2, linewidth=1,
)
cb = fig.colorbar(strm.lines, ax=ax)
cb.set_label("|B| (T)")
ax.set_xlabel("z (m)")
ax.set_ylabel("x (m)")
ax.set_title(f"WarpX B-field — iteration {its[-1]}")
ax.set_aspect("equal")
plt.show()

In [ ]:
# Constants from inputs/coil 2D/coil_2d.py (external_vector_potential.dipole):
#   K      vector-potential amplitude (T·m)
#   dh     half-separation between the two line currents (m)
#   eps    softening that keeps Ay finite on the line currents (m)
K   = 0.01
dh  = 0.25
eps = np.sqrt(0.006084)        # ≈ 0.078 m (matches eps² = 0.006084 in the run)


def Ay_analytic(x, z):
    """Static external A_y(x, z) used by the coil_2D run.

    Ax = Az = 0 by construction. Vectorised — `x` and `z` can be arrays of
    any matching shape.
    """
    return K * np.log(
        (x * x + (z + dh) ** 2 + eps * eps) /
        (x * x + (z - dh) ** 2 + eps * eps)
    )


# Build both axis coordinates from the openPMD metadata. The storage order is
# given by `axes` (= B.axis_labels); for this 2D Cartesian run it is
# ('z', 'x') — axis 0 is z, axis 1 is x. Map each physical coordinate to its
# real axis so xs/zs can never be silently swapped. (The earlier "missing
# negative-z bump" came from assuming ('x','z'): the plotted vertical axis was
# actually the flow/x axis, where the standoff is correctly one-sided.)
ix = list(axes).index("x")
iz = list(axes).index("z")
xs = x0[ix] * unit + (np.arange(shape[ix]) + B["x"].position[ix]) * dx[ix] * unit
zs = x0[iz] * unit + (np.arange(shape[iz]) + B["x"].position[iz]) * dx[iz] * unit
X, Z = np.meshgrid(xs, zs, indexing="ij")          # [x, z] layout

# Evaluate Ay on the grid and take its curl numerically. np.gradient uses
# second-order central differences in the interior — the same stencil
# class WarpX's Yee curl uses, so any deviation from the WarpX diagnostic
# at iteration 10 isolates the plasma-current contribution.
Ay_grid = Ay_analytic(X, Z)
dAy_dx = np.gradient(Ay_grid, xs, axis=0)
dAy_dz = np.gradient(Ay_grid, zs, axis=1)
Bx_an = -dAy_dz                                    # [x, z]
Bz_an = +dAy_dx                                    # [x, z]

mag_an = np.hypot(Bx_an, Bz_an).T

fig, ax = plt.subplots(figsize=(6, 5))
strm = ax.streamplot(
    xs, zs, Bx_an.T, Bz_an.T,
    color=mag_an, cmap="viridis", density=1.2, linewidth=1,
)
cb = fig.colorbar(strm.lines, ax=ax)
cb.set_label("|B| (T)")
ax.set_xlabel("x (m)")
ax.set_ylabel("z (m)")
ax.set_title("Analytic A → B = ∇×A   (expected match to WarpX iter 10)")
ax.set_aspect("equal")
plt.show()

# Quick numerical comparison against the WarpX snapshot already loaded
# above (Bx, Bz at iteration 10). The agreement won't be exact — 10
# hybrid steps have accumulated a small plasma-current contribution —
# but RMS rel error should be well under 10% if the grid and the formula
# match correctly. WarpX's Bx/Bz are stored [z, x]; transpose to [x, z] so
# the comparison is orientation-consistent with the analytic arrays.
def _rms_rel(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)) /
                 max(np.sqrt(np.mean(b ** 2)), 1e-30))

print(f"RMS rel error vs WarpX iter {its[1]}:")
print(f"  Bx : {_rms_rel(Bx_an, Bx.T):.3e}")
print(f"  Bz : {_rms_rel(Bz_an, Bz.T):.3e}")
print(f"  |B|: {_rms_rel(np.hypot(Bx_an, Bz_an), np.hypot(Bx, Bz).T):.3e}")


## Compare the thing to magpylib

In [ ]:
import magpylib as mp
from magpylib.current import Circle

bounds = 2.5 # m
ring_pos = (0,0,0)
ring_dia = 0.5 # m
ring_current = -1e5 # A

# make a circle coil aligned in the y-z plane
C = Circle(position=ring_pos, diameter=ring_dia, current=ring_current).rotate_from_angax(90, [0, 1, 0])

# Local coord array — kept _mag-suffixed so this cell never clobbers the
# `coords` defined above from the openPMD grid (which downstream β / leakage
# cells rely on).
coords_mag = np.linspace(-bounds, bounds, 100)

# get B-field from the grid
_grid = np.meshgrid(coords_mag, np.zeros(100), coords_mag, indexing='ij') # y-axis is always 0
grid = np.moveaxis(_grid, 0, -1)

B_mag_grid = C.getB(grid)
Bx_mag, By_mag, Bz_mag = np.moveaxis(B_mag_grid, -1, 0)

In [ ]:
Bx_slice = Bx_mag[:, 0, :]
Bz_slice = Bz_mag[:, 0, :]
mag = np.hypot(Bx_slice, Bz_slice).T

fig, ax = plt.subplots(figsize=(6, 5))
strm = ax.streamplot(
    coords_mag, coords_mag, Bx_slice.T, Bz_slice.T,
    color=mag, cmap="viridis", density=1.2, linewidth=1,
)
cb = fig.colorbar(strm.lines, ax=ax)
cb.set_label("|B| (T)")
ax.set_xlabel("x (m)")
ax.set_ylabel("z (m)")
ax.set_title("magpylib coil B-field (y = 0 slice)")
ax.set_aspect("equal")
plt.show()

In [ ]:
mp.show(C, backend="matplotlib")

## Pressure-balance diagnostics: β and the ram-pressure standoff

The coil_2D run is built to find the Chapman–Ferraro standoff — the surface where upstream ram pressure balances the dipole's magnetic pressure. Four pressures per cell carry the diagnostic:

- **Magnetic**: $p_{\text{mag}} = (B_x^2 + B_y^2 + B_z^2) / (2\mu_0)$ — direct from the field mesh.
- **Electron (fluid)**: $p_e = n_e T_e (n_e/n_0)^{\gamma-1}$ — $T_e$, $n_0$, $\gamma$ are solver inputs (must match `HybridPICSolver(Te=…, n0=…, gamma=…)` in the deck).
- **Ion thermal**: $p_{i,\text{th}} = \tfrac{1}{3} n_i m_i \sigma_v^2$ — needs the velocity variance per cell, binned from the particle dumps.
- **Ion ram**: $p_{\text{ram}} = n_i m_i \|\langle \mathbf{v} \rangle\|^2$ — needs the bulk velocity per cell, also from binning.

$n_i$ is taken from the deposited `rho_background_i + rho_stream_i` (better than re-binning — WarpX's deposition shape function is built in). $n_e = n_i$ by quasi-neutrality in hybrid PIC.

Plasma β is then $\beta_{\text{th}} = (p_{i,\text{th}} + p_e)/p_{\text{mag}}$ and $\beta_{\text{dyn}} = p_{\text{ram}}/p_{\text{mag}}$. The **$\beta_{\text{dyn}} = 1$ contour** is the empirical standoff surface — compare to the analytic `r_CF` printed at run start.

In [ ]:
import openpmd_api as io
import scipy.constants as sc

# --- Solver-side inputs that aren't in the diagnostic.
# Pulled straight from WarpX's used-inputs dump (written at run start) so they
# can never silently drift from the deck. Every resolved parameter is one
# `key = value` line in that file; parse the handful the pressure balance needs.
def _load_warpx_inputs(path="warpx_used_inputs"):
    params = {}
    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, val = line.split("=", 1)
            params[key.strip()] = val.strip()
    return params

_inp = _load_warpx_inputs()

mu0, m_i, q_i = sc.mu_0, sc.m_p, sc.e
clight = float(_inp["my_constants.clight"])

T_e_J   = float(_inp["hybrid_pic_model.elec_temp"]) * sc.eV   # elec_temp is in eV
gamma   = float(_inp["hybrid_pic_model.gamma"])
n0      = float(_inp["hybrid_pic_model.n0_ref"])              # closure ref density
n_floor = float(_inp["hybrid_pic_model.n_floor"])
# Stream drift speed for the ram-pressure reference. WarpX stores ux as
# proper-velocity / c (γβ); at these non-relativistic speeds v ≈ ux·c. The
# stream flows in −x, so take |ux| — only the magnitude feeds ρv².
v_drift = abs(float(_inp["stream_i.ux_m"])) * clight


# --- Velocity moments from particle dumps (both ion species combined) ---
def _bin_moments(species_list, xs, zs, iteration):
    """Return (⟨v⟩ per component, σ²_v summed) on the (xs, zs) grid for one
    openPMD iteration."""
    dxs, dzs = xs[1] - xs[0], zs[1] - zs[0]
    x_edges = np.r_[xs - 0.5 * dxs, xs[-1] + 0.5 * dxs]
    z_edges = np.r_[zs - 0.5 * dzs, zs[-1] + 0.5 * dzs]
    bins = [x_edges, z_edges]

    chunks = {k: [] for k in ("x", "z", "vx", "vy", "vz", "w")}
    for name in species_list:
        sp = iteration.particles[name]
        # openPMD load_chunk() returns a buffer that is only valid AFTER
        # series.flush(); arithmetic before the flush reads uninitialised
        # memory → spurious NaN/inf (this was the source of the earlier
        # "dropped non-finite particle(s)" / inf β warnings, not a real
        # solver blow-up). Capture the record components, load every chunk,
        # flush once, THEN convert to SI: positions are raw·unit_SI (→ m),
        # momenta are raw·unit_SI (→ kg·m/s) so v = raw·unit_SI / m_i.
        qx = sp["position"]["x"]; qz = sp["position"]["z"]
        px = sp["momentum"]["x"]; py = sp["momentum"]["y"]; pz = sp["momentum"]["z"]
        wt = sp["weighting"][io.Mesh_Record_Component.SCALAR]
        xr = qx.load_chunk(); zr = qz.load_chunk()
        pxr = px.load_chunk(); pyr = py.load_chunk(); pzr = pz.load_chunk()
        wr = wt.load_chunk()
        series.flush()
        chunks["x" ].append(xr.astype(np.float64) * qx.unit_SI)
        chunks["z" ].append(zr.astype(np.float64) * qz.unit_SI)
        chunks["vx"].append(pxr.astype(np.float64) * px.unit_SI / m_i)
        chunks["vy"].append(pyr.astype(np.float64) * py.unit_SI / m_i)
        chunks["vz"].append(pzr.astype(np.float64) * pz.unit_SI / m_i)
        chunks["w" ].append(wr.astype(np.float64))
    a = {k: np.concatenate(v) for k, v in chunks.items()}

    M0 = np.histogram2d(a["x"], a["z"], bins=bins, weights=a["w"])[0]
    safe = np.where(M0 > 0, M0, 1.0)
    vmean = []
    sigma2 = np.zeros_like(M0)
    for v in ("vx", "vy", "vz"):
        M1 = np.histogram2d(a["x"], a["z"], bins=bins, weights=a["w"] * a[v])[0]
        M2 = np.histogram2d(a["x"], a["z"], bins=bins, weights=a["w"] * a[v]**2)[0]
        mean = np.where(M0 > 0, M1 / safe, 0.0)
        sigma2 += np.where(M0 > 0, M2 / safe - mean**2, 0.0)
        vmean.append(mean)
    return vmean, sigma2


def compute_beta_dyn(iteration):
    """Full pressure-balance for one openPMD iteration → dict of [x, z] fields.

    WarpX 2D openPMD stores meshes as [z, x] (axis 0 = z, axis 1 = x). Every
    array returned here is transposed to [x, z] so the field-derived pressures
    (p_mag, p_e) share orientation with the histogram2d(x, z) particle moments
    (p_ram, p_i_th) — β = p_ram/p_mag is then not silently axis-swapped.
    Coordinates xs/zs come from the module scope (cell with the analytic A→B).
    """
    _B = iteration.meshes["B"]
    Bx_w = _B["x"].load_chunk()
    By_w = _B["y"].load_chunk()
    Bz_w = _B["z"].load_chunk()
    series.flush()
    Bx_w, By_w, Bz_w = Bx_w.T, By_w.T, Bz_w.T          # [z, x] → [x, z]

    def _rho(name):
        chunk = iteration.meshes[name][io.Mesh_Record_Component.SCALAR].load_chunk()
        series.flush()
        return chunk.T                                  # [z, x] → [x, z]

    rho_bg = _rho("rho_background_i")
    rho_st = _rho("rho_stream_i")
    n_i = (rho_bg + rho_st) / q_i
    n_e = n_i                                    # quasi-neutrality (hybrid)
    p_mag = (Bx_w**2 + By_w**2 + Bz_w**2) / (2.0 * mu0)
    n_e_safe = np.maximum(n_e, n_floor)
    p_e = n_e_safe * T_e_J * (n_e_safe / n0) ** (gamma - 1.0)

    vmean, sigma2 = _bin_moments(["background_i", "stream_i"], xs, zs, iteration)
    v_bulk_sq = vmean[0]**2 + vmean[1]**2 + vmean[2]**2

    # Ion pressures use n_i from the deposited rho fields (WarpX-shape-consistent),
    # not the histogram count. Moments only need ratios → normalisation-free.
    p_i_th = (1.0 / 3.0) * n_i * m_i * sigma2
    p_ram  =                n_i * m_i * v_bulk_sq

    p_mag_safe = np.maximum(p_mag, 1e-30)
    beta_th  = (p_i_th + p_e) / p_mag_safe
    beta_dyn = p_ram          / p_mag_safe
    return dict(beta_dyn=beta_dyn, beta_th=beta_th, p_mag=p_mag, p_e=p_e,
                p_i_th=p_i_th, p_ram=p_ram, n_i=n_i)


# Static snapshot at the last dumped iteration (drives the imshow cell below).
_bd = compute_beta_dyn(it)
beta_dyn, beta_th = _bd["beta_dyn"], _bd["beta_th"]
p_mag, p_e, p_i_th, p_ram, n_i = (_bd["p_mag"], _bd["p_e"], _bd["p_i_th"],
                                   _bd["p_ram"], _bd["n_i"])

# Sanity reference: undisturbed upstream ram pressure
P_ram_ref = n0 * m_i * v_drift ** 2
print(f"upstream ρv² (reference) = {P_ram_ref:.3e} Pa")
print(f"max p_mag                = {p_mag.max():.3e} Pa")
print(f"max p_e                  = {p_e.max():.3e} Pa")
print(f"max p_i_th               = {p_i_th.max():.3e} Pa")
print(f"max p_ram                = {p_ram.max():.3e} Pa")
print(f"max β_dyn                = {beta_dyn.max():.3e}")
print(f"max β_th                 = {beta_th.max():.3e}")

In [ ]:
extent = [xs[0], xs[-1], zs[0], zs[-1]]
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

# β_dyn heatmap + β_dyn = 1 standoff contour
im = axes[0].imshow(
    np.log10(np.maximum(beta_dyn, 1e-3)).T,
    origin="lower", extent=extent, cmap="coolwarm", vmin=-2, vmax=2,
)
axes[0].contour(xs, zs, beta_dyn.T, levels=[1.0], colors="k", linewidths=2)
fig.colorbar(im, ax=axes[0], label="log₁₀ β_dyn")
axes[0].set_title("β_dyn = ρv² / (B²/2μ₀)   (black contour: β_dyn=1, standoff)")
axes[0].set_xlabel("x (m)")
axes[0].set_ylabel("z (m)")
axes[0].set_aspect("equal")

# Pressure profiles along the stagnation line z ≈ 0
iz = int(np.argmin(np.abs(zs)))
axes[1].semilogy(xs, p_mag [:, iz], label="p_mag (B²/2μ₀)")
axes[1].semilogy(xs, p_e   [:, iz], label="p_e   (electron fluid)")
axes[1].semilogy(xs, p_i_th[:, iz], label="p_i_th (ion thermal)")
axes[1].semilogy(xs, p_ram [:, iz], label="p_ram (ion bulk flow)")
axes[1].set_xlabel("x (m)")
axes[1].set_ylabel("pressure (Pa)")
axes[1].set_title("Pressure-balance along stagnation line z ≈ 0")
axes[1].legend(loc="best")
axes[1].grid(True, which="both", alpha=0.3)

plt.show()

## Animated β_dyn + coupled midplane segment flux

The static map above is just the final dump. The cell below recomputes β_dyn for
**every** recorded diagnostic iteration (steps 0→1000) via `compute_beta_dyn` and
builds a coupled interactive Plotly animation:

- **Left** — a log₁₀ β_dyn heatmap (fixed color range −2..2, so frames are
  directly comparable) with the **β_dyn = 1 standoff contour** overlaid on each
  frame. Watch the Chapman–Ferraro cavity form over roughly one ion transit time
  (domain / v_drift ≈ step 490), settling symmetric in z and one-sided upstream in x.
- **Right** — the **midplane segment flux**: ions crossing the coil-to-coil line
  (`x=0, |z| < d/2`) in the downstream (−x) direction, ions·s⁻¹·m⁻¹. This is the
  non-perturbing flux counter written by `coil_2d.py` to `diags/segment_flux.npz`
  (a read-only callback — no absorbing wall, so the run is unperturbed). It
  measures the ion leak that penetrates the standoff into the inter-coil gap; it
  starts at the analytic background-drift level `n·v_drift·2d_h` and falls as the
  dipole deflects the stream. A red time-cursor tracks the currently playing
  frame, so the standoff state and the midplane leak are read at the same instant.

Drag the slider or hit ▶ to play both panels in lock-step.

In [ ]:
import os
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Inline plotly.js into the saved notebook so the animation stays interactive
# offline (vs "notebook_connected", which fetches plotly.js from a CDN).
pio.renderers.default = "notebook"

# --- Recompute β_dyn for every dumped iteration ---------------------------
# compute_beta_dyn returns [x, z]; the static cells plot beta.T (so x is the
# horizontal axis, z vertical). Build the same orientation per frame and stack.
_steps = sorted(series.iterations)
lb_frames = []          # log10 β_dyn, [z, x] (Heatmap/Contour orientation)
times_us  = []
for n in _steps:
    _itn = series.iterations[n]
    bd = compute_beta_dyn(_itn)["beta_dyn"]
    lb_frames.append(np.log10(np.maximum(bd, 1e-3)).T)   # [x,z] → [z,x]
    times_us.append(float(_itn.time) * float(_itn.time_unit_SI) * 1e6)
lb_stack = np.stack(lb_frames)                            # (n_iter, z, x)

# --- midplane segment flux rate vs time -----------------------------------
# coil_2d.py's non-perturbing flux counter writes ions·s⁻¹·m⁻¹ crossing the
# coil-to-coil segment (x=0, |z|<dh) each PIC step to diags/segment_flux.npz.
# `rate_minus` is the downstream-going (−x) leak that penetrates the standoff —
# load it directly (no openPMD binning). Re-run coil_2d.py to (re)generate it.
_seg_path = "diags/segment_flux.npz"
if os.path.exists(_seg_path):
    _seg = np.load(_seg_path)
    leak_t = _seg["t_s"] * 1e6                 # µs
    leak_rate = _seg["rate_minus"]             # ions · s⁻¹ · m⁻¹
else:
    print(f"[warn] {_seg_path} not found — re-run coil_2d.py to populate the "
          f"segment flux counter. Right panel shows a flat placeholder.")
    leak_t = np.array([0.0, times_us[-1]])
    leak_rate = np.zeros(2)
# Log-y on the flux panel: the early background-drift level (≈ n·v·2dₕ) is a
# large multiple of the suppressed late-time tail, so a linear axis buries the
# standoff-formation window.
_pos = leak_rate[leak_rate > 0]
_ymax = max(float(leak_rate.max()) * 1.3, 10.0)
_yfloor = max(float(_pos.min()) * 0.5, 1.0) if _pos.size else 1.0

# --- Trace / cursor builders ----------------------------------------------
ZMIN, ZMAX = -2.0, 2.0
_contour = dict(start=0.0, end=0.0, size=1.0, coloring="none")  # β_dyn=1 line


def _base_left(lb):
    """FULL heatmap + β=1 contour traces for the base figure (heavy style set
    once and persists)."""
    return [
        go.Heatmap(z=lb, x=xs, y=zs, zmin=ZMIN, zmax=ZMAX, colorscale="RdBu_r",
                   colorbar=dict(title="log₁₀ β_dyn", x=0.52, len=0.9,
                                 thickness=12),
                   zsmooth=False, xaxis="x", yaxis="y"),
        go.Contour(z=lb, x=xs, y=zs, contours=_contour,
                   line=dict(color="black", width=2), showscale=False,
                   hoverinfo="skip", xaxis="x", yaxis="y"),
    ]


def _cursor_shape(t_us):
    """Time-cursor as a LAYOUT shape (not a trace). Keeping the pointer out of
    the frame *data* means frames carry only the (non-transitionable) heatmap +
    contour — exactly like the single-panel v1 — so Plotly redraws each frame
    promptly without the heatmap blanking (a transitionable scatter in the
    frame forced that) and without the hard-redraw flash that transition=0 added.
    yref='y2 domain' spans the panel full-height regardless of the log y-scale."""
    return dict(type="line", xref="x2", yref="y2 domain",
                x0=t_us, x1=t_us, y0=0, y1=1,
                line=dict(color="crimson", width=2, dash="dash"))


# --- Assemble coupled figure (heatmap | segment flux) ---------------------
fig = make_subplots(
    rows=1, cols=2, column_widths=[0.58, 0.42], horizontal_spacing=0.12,
    subplot_titles=("β_dyn = ρv² / (B²/2μ₀)   (black: β_dyn=1 standoff)",
                    "midplane segment flux (−x)"),
)
# Base traces (index order: 0 heatmap, 1 contour, 2 flux curve).
for tr in _base_left(lb_stack[0]):
    fig.add_trace(tr, row=1, col=1)
fig.add_trace(go.Scatter(x=leak_t, y=leak_rate, mode="lines",
                         line=dict(color="steelblue", width=1.8),
                         name="segment −x rate", showlegend=False),
              row=1, col=2)

# Frames update ONLY the heatmap + contour z (traces 0,1) — like v1 — while the
# cursor rides along as a per-frame layout shape.
fig.frames = [
    go.Frame(data=[go.Heatmap(z=lb), go.Contour(z=lb)], traces=[0, 1],
             layout=go.Layout(shapes=[_cursor_shape(t)]), name=str(n))
    for n, lb, t in zip(_steps, lb_stack, times_us)
]

# v1 animation args (no transition override): redraw=True repaints the heatmap z,
# default transition + frame-only data keeps playback smooth (no flash, no blank).
_play = dict(label="▶ Play", method="animate",
             args=[None, dict(frame=dict(duration=120, redraw=True),
                              fromcurrent=True, mode="immediate")])
_pause = dict(label="⏸ Pause", method="animate",
              args=[[None], dict(frame=dict(duration=0, redraw=False),
                                 mode="immediate")])
_slider = dict(
    active=0, x=0.05, len=0.95, pad=dict(t=45),
    currentvalue=dict(prefix="step ", visible=True),
    steps=[dict(method="animate", label=str(n),
                args=[[str(n)], dict(frame=dict(duration=0, redraw=True),
                                     mode="immediate")])
           for n in _steps],
)

fig.update_yaxes(scaleanchor="x", scaleratio=1, title_text="z (m)", row=1, col=1)
fig.update_xaxes(title_text="x (m)", constrain="domain", row=1, col=1)
fig.update_xaxes(title_text="time (µs)", range=[0, times_us[-1]], row=1, col=2)
fig.update_yaxes(title_text="ions · s⁻¹·m⁻¹ (log)", type="log",
                 range=[np.log10(_yfloor), np.log10(_ymax)], row=1, col=2)
fig.update_layout(
    width=1180, height=620, margin=dict(t=90),
    shapes=[_cursor_shape(times_us[0])],
    updatemenus=[dict(type="buttons", direction="left", showactive=False,
                      x=0.05, y=1.16, xanchor="left", buttons=[_play, _pause])],
    sliders=[_slider],
)

print(f"frames={len(fig.frames)}  steps {_steps[0]}→{_steps[-1]}  "
      f"t {times_us[0]:.3f}→{times_us[-1]:.3f} µs  "
      f"segment −x peak rate={leak_rate.max():.3e} ions·s⁻¹·m⁻¹")
fig.show()

## Animated plasma density + cusp density

Same coupled-animation recipe as the β_dyn cell above, but for the **plasma density**:

- **Left** — animated log₁₀ n_i heatmap (fixed color range across frames, so they are
  directly comparable). n_i comes straight from the deposited
  `rho_background_i + rho_stream_i` meshes — no particle binning — so this sweep is
  much lighter than the β_dyn one. The white dashed segment marks the cusp.
- **Right** — mean n_i **in the cusp**: the coil-to-coil midplane segment the flux
  counter watches (`x = 0, |z| < dₕ`), sampled as the cell column straddling `x = 0`.
  One point per diagnostic dump. The red time-cursor tracks the currently playing
  frame so the density map and the cusp fill state are read at the same instant.

Drag the slider or hit ▶ to play both panels in lock-step.


In [ ]:
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "notebook"


# --- n_i per dumped iteration ----------------------------------------------
# Density needs only the deposited rho meshes (no particle binning), so this
# sweep is much lighter than the β_dyn one. Same [z, x] → [x, z] transpose
# convention as compute_beta_dyn.
def _n_i_field(iteration):
    """Ion density n_i = (rho_background_i + rho_stream_i)/q_i, [x, z] (m⁻³)."""
    def _rho(name):
        chunk = iteration.meshes[name][io.Mesh_Record_Component.SCALAR].load_chunk()
        series.flush()
        return chunk.T                                  # [z, x] → [x, z]
    return (_rho("rho_background_i") + _rho("rho_stream_i")) / q_i


# Cusp = the coil-to-coil midplane segment the flux counter in coil_2d.py
# watches (x = 0, |z| < dh). Sample the cell column(s) straddling x = 0 —
# cell centers sit at ±dx/2, so |x| < dx catches exactly the two nearest.
_dx_cell = xs[1] - xs[0]
_mx_cusp = np.abs(xs) < _dx_cell
_mz_cusp = np.abs(zs) < dh

_steps_n  = sorted(series.iterations)
ln_frames = []            # log10 n_i, [z, x] (Heatmap orientation)
ni_times_us = []
cusp_n = []               # mean n_i in the cusp per frame
for n in _steps_n:
    _itn = series.iterations[n]
    n_i_f = _n_i_field(_itn)
    ln_frames.append(np.log10(np.maximum(n_i_f, n_floor)).T)   # [x,z] → [z,x]
    cusp_n.append(float(n_i_f[np.ix_(_mx_cusp, _mz_cusp)].mean()))
    ni_times_us.append(float(_itn.time) * float(_itn.time_unit_SI) * 1e6)
ln_stack = np.stack(ln_frames)                                 # (n_iter, z, x)
cusp_n = np.array(cusp_n)

# Fixed color range across frames so the animation is directly comparable.
NZMIN = float(np.floor(ln_stack.min()))
NZMAX = float(np.ceil(ln_stack.max()))

# Log-y range for the cusp panel (density can fall orders of magnitude as the
# standoff forms — linear would bury the late-time tail).
_ny_max = float(cusp_n.max()) * 1.5
_ny_min = (max(float(cusp_n[cusp_n > 0].min()) * 0.5, n_floor)
           if (cusp_n > 0).any() else n_floor)


def _cursor_shape_n(t_us):
    """Time-cursor as a LAYOUT shape on the right panel — same trick as the
    β_dyn animation: keeping the pointer out of the frame *data* means frames
    carry only the heatmap z, so playback never blanks or flashes."""
    return dict(type="line", xref="x2", yref="y2 domain",
                x0=t_us, x1=t_us, y0=0, y1=1,
                line=dict(color="crimson", width=2, dash="dash"))


# --- Assemble coupled figure (density heatmap | cusp density) --------------
fig_n = make_subplots(
    rows=1, cols=2, column_widths=[0.58, 0.42], horizontal_spacing=0.12,
    subplot_titles=("log₁₀ n_i (m⁻³)   (white dash: cusp segment)",
                    "mean n_i in the cusp (x=0, |z|<dₕ)"),
)
# Base traces (index order: 0 heatmap [animated], 1 cusp marker, 2 density curve).
fig_n.add_trace(
    go.Heatmap(z=ln_stack[0], x=xs, y=zs, zmin=NZMIN, zmax=NZMAX,
               colorscale="Viridis",
               colorbar=dict(title="log₁₀ n_i", x=0.52, len=0.9, thickness=12),
               zsmooth=False),
    row=1, col=1)
# Static marker of the sampled cusp segment — its own trace (not in frames),
# so it persists through playback without being re-sent every frame.
fig_n.add_trace(
    go.Scatter(x=[0.0, 0.0], y=[-dh, dh], mode="lines",
               line=dict(color="white", width=2, dash="dash"),
               showlegend=False, hoverinfo="skip"),
    row=1, col=1)
fig_n.add_trace(
    go.Scatter(x=ni_times_us, y=cusp_n, mode="lines+markers",
               line=dict(color="steelblue", width=1.8), marker=dict(size=4),
               showlegend=False),
    row=1, col=2)

# Frames update ONLY the heatmap z (trace 0); the cursor rides along as a
# per-frame layout shape — identical recipe to the β_dyn animation above.
fig_n.frames = [
    go.Frame(data=[go.Heatmap(z=ln)], traces=[0],
             layout=go.Layout(shapes=[_cursor_shape_n(t)]), name=str(n))
    for n, ln, t in zip(_steps_n, ln_stack, ni_times_us)
]

_play_n = dict(label="▶ Play", method="animate",
               args=[None, dict(frame=dict(duration=120, redraw=True),
                                fromcurrent=True, mode="immediate")])
_pause_n = dict(label="⏸ Pause", method="animate",
                args=[[None], dict(frame=dict(duration=0, redraw=False),
                                   mode="immediate")])
_slider_n = dict(
    active=0, x=0.05, len=0.95, pad=dict(t=45),
    currentvalue=dict(prefix="step ", visible=True),
    steps=[dict(method="animate", label=str(n),
                args=[[str(n)], dict(frame=dict(duration=0, redraw=True),
                                     mode="immediate")])
           for n in _steps_n],
)

fig_n.update_yaxes(scaleanchor="x", scaleratio=1, title_text="z (m)", row=1, col=1)
fig_n.update_xaxes(title_text="x (m)", constrain="domain", row=1, col=1)
fig_n.update_xaxes(title_text="time (µs)", range=[0, ni_times_us[-1]], row=1, col=2)
fig_n.update_yaxes(title_text="n_i (m⁻³, log)", type="log",
                   range=[np.log10(_ny_min), np.log10(_ny_max)], row=1, col=2)
fig_n.update_layout(
    width=1180, height=620, margin=dict(t=90),
    shapes=[_cursor_shape_n(ni_times_us[0])],
    updatemenus=[dict(type="buttons", direction="left", showactive=False,
                      x=0.05, y=1.16, xanchor="left",
                      buttons=[_play_n, _pause_n])],
    sliders=[_slider_n],
)

print(f"frames={len(fig_n.frames)}  steps {_steps_n[0]}→{_steps_n[-1]}  "
      f"t {ni_times_us[0]:.3f}→{ni_times_us[-1]:.3f} µs  "
      f"cusp ⟨n_i⟩ {cusp_n[0]:.3e} → {cusp_n[-1]:.3e} m⁻³")
fig_n.show()


### Export the density animation as a video file

Plotly animations live in the browser — there is no built-in video export. This cell
re-renders the same frames (`ln_stack`, `cusp_n`, `ni_times_us` from the cell above)
with matplotlib and writes a standalone file to `diags/`:

- **GIF** via Pillow (works with the current env, no extra deps), or
- **MP4** via ffmpeg automatically, if you `conda install -c conda-forge ffmpeg`.


In [ ]:
import os
import sys
import matplotlib.pyplot as plt
import matplotlib.animation as manim

# MP4 needs ffmpeg. Prefer the conda env's own binary (env bin isn't always on
# the kernel's PATH, e.g. under PyCharm); without it, fall back to a Pillow GIF.
_env_ffmpeg = os.path.join(sys.prefix, "bin", "ffmpeg")
if not manim.writers.is_available("ffmpeg") and os.path.exists(_env_ffmpeg):
    plt.rcParams["animation.ffmpeg_path"] = _env_ffmpeg
_have_ffmpeg = manim.writers.is_available("ffmpeg")
_vid_out = os.path.join("diags", "density_anim.mp4" if _have_ffmpeg else "density_anim.gif")

fig_v, (axL, axR) = plt.subplots(
    1, 2, figsize=(12, 5.2), constrained_layout=True,
    gridspec_kw=dict(width_ratios=[1.25, 1]),
)
im_v = axL.imshow(ln_stack[0], origin="lower",
                  extent=[xs[0], xs[-1], zs[0], zs[-1]],
                  cmap="viridis", vmin=NZMIN, vmax=NZMAX, interpolation="nearest")
fig_v.colorbar(im_v, ax=axL, label="log₁₀ n_i (m⁻³)")
axL.plot([0, 0], [-dh, dh], "w--", lw=2)
axL.set_xlabel("x (m)")
axL.set_ylabel("z (m)")
axL.set_aspect("equal")
title_v = axL.set_title("")

axR.plot(ni_times_us, cusp_n / 1e17, color="steelblue", lw=1.8)
cursor_v = axR.axvline(ni_times_us[0], color="crimson", ls="--", lw=2)
dot_v, = axR.plot([ni_times_us[0]], [cusp_n[0] / 1e17], "o", color="crimson", ms=6)
axR.set_xlabel("time (µs)")
axR.set_ylabel("⟨n_i⟩ cusp (10¹⁷ m⁻³)")
axR.set_title("mean n_i in the cusp (x=0, |z|<dₕ)")
axR.grid(alpha=0.3)


def _update_v(i):
    im_v.set_data(ln_stack[i])
    cursor_v.set_xdata([ni_times_us[i]] * 2)
    dot_v.set_data([ni_times_us[i]], [cusp_n[i] / 1e17])
    title_v.set_text(f"log₁₀ n_i — step {_steps_n[i]}, t = {ni_times_us[i]:.2f} µs")
    return im_v, cursor_v, dot_v, title_v


ani_v = manim.FuncAnimation(fig_v, _update_v, frames=len(_steps_n), blit=False)
if _have_ffmpeg:
    ani_v.save(_vid_out, writer=manim.FFMpegWriter(fps=8, bitrate=2000), dpi=100)
else:
    ani_v.save(_vid_out, writer=manim.PillowWriter(fps=8), dpi=80)
plt.close(fig_v)
print(f"wrote {_vid_out}  ({os.path.getsize(_vid_out)/1e6:.1f} MB, "
      f"{len(_steps_n)} frames @ 8 fps)")


## Boundary leakage rate vs time

The deck configures a `ParticleBoundaryScrapingDiagnostic` (`name="scrape"`) that records every macroparticle absorbed at the four field boundaries (`x_lo`, `x_hi`, `z_lo`, `z_hi`), writing to a **separate** openPMD series at `diags/scrape/openpmd_%T.h5`. Each scraped particle carries:

| Record | What it's for |
|---|---|
| `position.{x, z}` | Which face it crossed (mask by threshold near each edge) |
| `momentum.{x, y, z}` | Energy-loss diagnostics if you want them (`E_kin = ½ m_i \|v\|²`) |
| `weighting` | Physical particles per macroparticle — what to **sum** for a count |
| `stepScraped` | PIC step at which the particle was removed — divide by `const_dt` for time |

The output is flushed every `PERIOD = 10` steps, but each particle's `stepScraped` gives per-step time resolution regardless.

Three plots out of one aggregation pass:

1. **Total leakage rate per species** — `background_i` vs `stream_i`, particles/s vs time.
2. **Per-face stacked rate** — which boundary is doing the leaking. For coil_2D, expect `x_lo` (downstream of the stream) to dominate after the front transits; `z_lo` / `z_hi` light up when cusp-channel losses appear.
3. **Cumulative leakage** — sanity-check vs the injected flux `n_stream · v_drift · A_inject · t`. Steady state ⇔ cumulative loss tracks injection.

The cell below is **defensive**: if `diags/scrape/` doesn't exist yet (no run has written it), it prints a hint and skips, so the rest of the notebook keeps executing.

In [ ]:
import os
import glob
import openpmd_api as io
import scipy.constants as sc

# Physical domain extents from the deck — only used for the injected-flux
# reference in the plot cell. The "which face?" classification no longer
# relies on position thresholds: WarpX's BoundaryScrapingDiagnostic writes
# one subdirectory per boundary, so the directory name IS the face.
Lx_DOMAIN = 2.5
Lz_DOMAIN = 2.5

SPECIES = ("background_i", "stream_i")
SCRAPE_DIR = "diags/scrape"

# WarpX's BoundaryScrapingDiagnostic with openPMD format writes one
# subdirectory per active boundary, each its own `openpmd_%T.h5` series:
#   diags/scrape/particles_at_xlo/openpmd_*.h5
#   diags/scrape/particles_at_xhi/openpmd_*.h5
#   diags/scrape/particles_at_zlo/openpmd_*.h5
#   diags/scrape/particles_at_zhi/openpmd_*.h5
# Map each canonical face label to its subdirectory.
BOUNDARY_DIRS = {
    "x_lo": "particles_at_xlo",
    "x_hi": "particles_at_xhi",
    "z_lo": "particles_at_zlo",
    "z_hi": "particles_at_zhi",
}
FACES = tuple(BOUNDARY_DIRS)


def _load_component(record_component):
    """Load an openPMD record component as float64, or None if it's empty.

    Empty species in a given iteration are written as constant/zero-extent
    records that have no real data to chunk-load — return None for those.
    """
    try:
        if getattr(record_component, "empty", False):
            return None
        chunk = record_component.load_chunk()
    except (KeyError, IndexError, RuntimeError):
        return None
    return chunk


def _aggregate_scrape(scrape_dir, species_list):
    """Walk every boundary subdirectory and concatenate scraped particles
    per species, tagging each particle with the face it crossed (from the
    subdirectory it came from).

    Returns dict[species] → dict of arrays (incl. a "face" label array)
    plus the common dt pulled from the openPMD iteration attribute.
    """
    buf = {s: {k: [] for k in ("x", "z", "vx", "vy", "vz", "w", "step", "face")}
           for s in species_list}
    dt = None

    for face, subdir in BOUNDARY_DIRS.items():
        pattern = os.path.join(scrape_dir, subdir, "openpmd_%T.h5")
        if not glob.glob(os.path.join(scrape_dir, subdir, "openpmd_*.h5")):
            continue
        series = io.Series(pattern, io.Access.read_only)
        for it_idx in sorted(series.iterations):
            iteration = series.iterations[it_idx]
            if dt is None:
                dt = float(iteration.dt)
            for sp_name in species_list:
                if sp_name not in iteration.particles:
                    continue
                sp = iteration.particles[sp_name]
                x  = _load_component(sp["position"]["x"])
                z  = _load_component(sp["position"]["z"])
                px = _load_component(sp["momentum"]["x"])
                py = _load_component(sp["momentum"]["y"])
                pz = _load_component(sp["momentum"]["z"])
                w  = _load_component(
                    sp["weighting"][io.Mesh_Record_Component.SCALAR])
                step = _load_component(
                    sp["stepScraped"][io.Mesh_Record_Component.SCALAR])
                series.flush()
                # Nothing scraped for this species this period.
                if x is None or len(x) == 0 or w is None:
                    continue
                buf[sp_name]["x"].append(x.astype(np.float64))
                buf[sp_name]["z"].append(z.astype(np.float64))
                buf[sp_name]["vx"].append(px.astype(np.float64) / sc.m_p)
                buf[sp_name]["vy"].append(py.astype(np.float64) / sc.m_p)
                buf[sp_name]["vz"].append(pz.astype(np.float64) / sc.m_p)
                buf[sp_name]["w"].append(w.astype(np.float64))
                buf[sp_name]["step"].append(step.astype(np.float64))
                buf[sp_name]["face"].append(np.full(len(x), face))

    out = {}
    for sp_name, d in buf.items():
        if not d["x"]:
            out[sp_name] = None
            continue
        out[sp_name] = {k: np.concatenate(v) for k, v in d.items()}
        out[sp_name]["time"] = out[sp_name]["step"].astype(float) * dt
    return out, dt


def _scrape_has_data(scrape_dir):
    """True iff at least one boundary subdir holds an openPMD series."""
    return any(
        glob.glob(os.path.join(scrape_dir, subdir, "openpmd_*.h5"))
        for subdir in BOUNDARY_DIRS.values()
    )


# ---- Defensive load ------------------------------------------------------
if not os.path.isdir(SCRAPE_DIR) or not _scrape_has_data(SCRAPE_DIR):
    # Most common cause: the species in inputs/coil 2D/coil_2d.py are missing
    # the `warpx_save_particles_at_<face>=1` flags. The diagnostic itself
    # being configured is necessary but not sufficient — the per-species
    # flags populate the buffer that the diagnostic flushes.
    print(f"[skip] no scrape output found at {SCRAPE_DIR}/.")
    print("       Possible causes (in order of likelihood):")
    print(f"         1. {SCRAPE_DIR}/particles_at_*/ does not exist yet — "
          f"re-run coil_2d.py")
    print(f"            with the species flags ("
          f"warpx_save_particles_at_{{xlo,xhi,zlo,zhi}}=1).")
    print(f"         2. Output layout differs from the expected per-boundary "
          f"`particles_at_*/openpmd_%T.h5` series.")
    scraped = None
    dt_run  = None
else:
    scraped, dt_run = _aggregate_scrape(SCRAPE_DIR, SPECIES)
    total_macros = sum(0 if d is None else len(d["w"]) for d in scraped.values())
    total_phys   = sum(0 if d is None else d["w"].sum() for d in scraped.values())
    t_max_us = max((d["time"].max() for d in scraped.values() if d is not None),
                   default=0.0) * 1e6
    print(f"const_dt              = {dt_run:.3e} s")
    print(f"scraped macroparticles = {total_macros}")
    print(f"scraped physical ions  = {total_phys:.3e}")
    print(f"final t (scraped)      = {t_max_us:.3f} µs")


# ---- Bin scrape events into time histograms -----------------------------
def _binned_rates(scraped, dt, n_bins=80):
    """For each species, return rate(t) total + per-face."""
    if scraped is None:
        return None
    # Common bin edges across all species so the time axes line up
    t_max = max(d["time"].max() for d in scraped.values() if d is not None)
    t_edges = np.linspace(0.0, max(t_max, dt), n_bins + 1)
    bin_dt  = t_edges[1] - t_edges[0]
    t_centers = 0.5 * (t_edges[:-1] + t_edges[1:])

    total = {}
    per_face = {}
    cumulative = {}
    for sp, d in scraped.items():
        if d is None:
            total[sp] = np.zeros_like(t_centers)
            per_face[sp] = {f: np.zeros_like(t_centers) for f in FACES}
            cumulative[sp] = np.zeros_like(t_centers)
            continue
        hist, _ = np.histogram(d["time"], bins=t_edges, weights=d["w"])
        total[sp] = hist / bin_dt
        cumulative[sp] = np.cumsum(hist)
        per_face[sp] = {}
        for fname in FACES:
            m = d["face"] == fname
            h, _ = np.histogram(d["time"][m], bins=t_edges, weights=d["w"][m])
            per_face[sp][fname] = h / bin_dt

    return t_centers, t_edges, total, per_face, cumulative


leakage = _binned_rates(scraped, dt_run) if scraped is not None else None

In [ ]:
if leakage is None:
    print("[skip] no leakage data — re-run the notebook after the simulation "
          "has written diags/scrape/.")
else:
    t_centers, t_edges, total, per_face, cumulative = leakage
    t_us = t_centers * 1e6

    fig, axes = plt.subplots(3, 1, figsize=(10, 11),
                             sharex=True, constrained_layout=True)

    # ---- (1) Total leakage rate per species ---------------------------------
    for sp in SPECIES:
        axes[0].plot(t_us, total[sp], label=sp, linewidth=1.5)
    axes[0].set_ylabel("rate (particles · s⁻¹)")
    axes[0].set_title("Total leakage rate vs time")
    axes[0].set_yscale("symlog", linthresh=1.0)
    axes[0].grid(alpha=0.3)
    axes[0].legend(loc="best")

    # ---- (2) Per-face stacked rate (both species combined) ------------------
    faces = ("x_lo", "x_hi", "z_lo", "z_hi")
    face_total = {f: sum(per_face[sp][f] for sp in SPECIES) for f in faces}
    axes[1].stackplot(t_us,
                      [face_total[f] for f in faces],
                      labels=faces, alpha=0.7)
    axes[1].set_ylabel("rate (particles · s⁻¹)")
    axes[1].set_title("Per-face leakage rate (sum of species)")
    axes[1].grid(alpha=0.3)
    axes[1].legend(loc="upper right")

    # ---- (3) Cumulative leakage, with injection reference -------------------
    for sp in SPECIES:
        axes[2].plot(t_us, cumulative[sp], label=f"cumulative {sp}",
                     linewidth=1.5)
    # Reference: undisturbed injected flux at the x_hi face. flux × area × t.
    # 2D area = 2·Lz × 1 m (the suppressed y-direction). n·v must match the
    # deck stream params in coil_2d.py (n_stream=1e19, v_drift=5e4).
    injected_ref = 1.0e19 * 5.0e4 * (2.0 * Lz_DOMAIN) * t_centers
    axes[2].plot(t_us, injected_ref, "k--",
                 label="injected (n·v·A·t)", linewidth=1.0)
    axes[2].set_xlabel("time (µs)")
    axes[2].set_ylabel("particles (cumulative)")
    axes[2].set_title("Cumulative leakage vs injected flux")
    axes[2].grid(alpha=0.3)
    axes[2].legend(loc="best")

    # Top x-axis: PIC step number (step = time / const_dt)
    def t_to_step(t):     return t * 1e-6 / dt_run
    def step_to_t(step):  return step * dt_run * 1e6
    sec_ax = axes[0].secondary_xaxis("top", functions=(t_to_step, step_to_t))
    sec_ax.set_xlabel("PIC step")

    plt.show()